# Guided Lab: Text-to-SQL Agent (SQLite Edition)

This lab shows a basic text-to-SQL setup using:

- a local file-based SQLite database
- a mock application dataset
- a small SQL agent pattern
- optional LLM backends: Groq, OpenAI, or Ollama
- LangSmith tracing for visibility

LangChain’s SQL-agent guide shows the core flow: inspect tables and schemas, choose the relevant tables, generate a query, double-check it, execute it, and answer from the result. The LangSmith custom store guide shows how a store can be provided through an async context manager, and it notes that SQLite is not recommended for production deployments.


## Learning goals

By the end of this lab, you should be able to:

1. Create a small SQLite database file.
2. Seed mock application tables.
3. Inspect the database schema.
4. Route natural-language questions to SQL.
5. Connect the lab to Groq, OpenAI, or Ollama.
6. Turn on LangSmith tracing.


## 1) Install packages

Install the basic dependencies first.

```bash
pip install -U python-dotenv sqlalchemy sqlmodel langchain langgraph langsmith langchain-community langchain-groq langchain-openai langchain-ollama
```

For this lab, SQLite is used as the local database. The SQL agent tutorial explicitly recommends narrow database permissions because model-generated SQL carries risk.


In [1]:
%pip install -qU python-dotenv sqlalchemy sqlmodel langchain langgraph langsmith langchain-community langchain-groq langchain-openai langchain-ollama


Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 3.10.1 requires pandas<3, but you have pandas 3.0.3 which is incompatible.
sagemaker-serve 1.5.0 requires sagemaker-core>=2.5.0, but you have sagemaker-core 1.0.77 which is incompatible.
sagemaker-train 1.5.0 requires sagemaker-core>=2.5.0, but you have sagemaker-core 1.0.77 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Environment variables

Create environment-specific values in `.env` or set them in your shell.

```env
APP_ENV=dev
DB_PATH=lab_app.db
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=text-to-sql-lab

MODEL_PROVIDER=groq
MODEL_NAME=llama-3.3-70b-versatile
```

You can switch `MODEL_PROVIDER` to `openai` or `ollama` later.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

APP_ENV = os.getenv("APP_ENV", "dev")
DB_PATH = os.getenv("DB_PATH", "lab_app.db")
MODEL_PROVIDER = os.getenv("MODEL_PROVIDER", "groq").lower()
MODEL_NAME = os.getenv("MODEL_NAME", "llama-3.3-70b-versatile")

print("APP_ENV:", APP_ENV)
print("DB_PATH:", DB_PATH)
print("MODEL_PROVIDER:", MODEL_PROVIDER)
print("MODEL_NAME:", MODEL_NAME)
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))


APP_ENV: dev
DB_PATH: lab_app.db
MODEL_PROVIDER: groq
MODEL_NAME: llama-3.3-70b-versatile
LANGSMITH_TRACING: true


## 3) Create a mock application database

This lab uses a tiny e-commerce style dataset so the SQL examples are easy to understand.

Tables:
- customers
- orders
- order_items
- support_tickets


In [3]:
import sqlite3
from pathlib import Path

db_file = Path(DB_PATH)

# Start fresh for the lab
if db_file.exists():
    db_file.unlink()

conn = sqlite3.connect(db_file)
cur = conn.cursor()

cur.execute('CREATE TABLE customers (customer_id INTEGER PRIMARY KEY AUTOINCREMENT, full_name TEXT NOT NULL, city TEXT NOT NULL, signup_date TEXT NOT NULL)')
cur.execute('CREATE TABLE orders (order_id INTEGER PRIMARY KEY AUTOINCREMENT, customer_id INTEGER NOT NULL, order_date TEXT NOT NULL, status TEXT NOT NULL, total_amount REAL NOT NULL, FOREIGN KEY (customer_id) REFERENCES customers(customer_id))')
cur.execute('CREATE TABLE order_items (item_id INTEGER PRIMARY KEY AUTOINCREMENT, order_id INTEGER NOT NULL, product_name TEXT NOT NULL, quantity INTEGER NOT NULL, unit_price REAL NOT NULL, FOREIGN KEY (order_id) REFERENCES orders(order_id))')
cur.execute('CREATE TABLE support_tickets (ticket_id INTEGER PRIMARY KEY AUTOINCREMENT, customer_id INTEGER NOT NULL, subject TEXT NOT NULL, status TEXT NOT NULL, created_at TEXT NOT NULL, FOREIGN KEY (customer_id) REFERENCES customers(customer_id))')

conn.commit()
conn.close()

print("Created database:", db_file.resolve())


Created database: D:\personal_docs\course-ai\module-1\db\lab_app.db


## 4) Seed mock data

The dataset is intentionally small so you can focus on the SQL workflow.


In [4]:
def seed_data(db_path: str):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    customers = [
        ("Asha Rao", "Chennai", "2026-01-02"),
        ("Imran Khan", "Bengaluru", "2026-01-10"),
        ("Neha Patel", "Mumbai", "2026-01-18"),
    ]
    cur.executemany("INSERT INTO customers (full_name, city, signup_date) VALUES (?, ?, ?)", customers)

    orders = [
        (1, "2026-02-01", "completed", 2499.0),
        (1, "2026-02-12", "completed", 899.0),
        (2, "2026-02-14", "processing", 1499.0),
        (3, "2026-02-20", "completed", 3199.0),
    ]
    cur.executemany("INSERT INTO orders (customer_id, order_date, status, total_amount) VALUES (?, ?, ?, ?)", orders)

    items = [
        (1, "Keyboard", 1, 2499.0),
        (2, "Mouse", 1, 899.0),
        (3, "Headphones", 1, 1499.0),
        (4, "Monitor", 1, 3199.0),
    ]
    cur.executemany("INSERT INTO order_items (order_id, product_name, quantity, unit_price) VALUES (?, ?, ?, ?)", items)

    tickets = [
        (1, "Refund request", "open", "2026-02-03T09:00:00"),
        (2, "Order status question", "closed", "2026-02-15T10:30:00"),
        (3, "Invoice needed", "open", "2026-02-21T08:45:00"),
    ]
    cur.executemany("INSERT INTO support_tickets (customer_id, subject, status, created_at) VALUES (?, ?, ?, ?)", tickets)

    conn.commit()
    conn.close()

seed_data(DB_PATH)
print("Seeded mock data.")


Seeded mock data.


## 5) Inspect the database

Before asking questions, always look at the tables and schema first. That matches the LangChain SQL-agent workflow. citeturn845823view0


In [5]:
def list_tables(db_path: str) -> list[str]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = [row[0] for row in cur.fetchall() if not row[0].startswith('sqlite_')]
    conn.close()
    return tables

def get_schema(db_path: str, table_name: str) -> str:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute(f'PRAGMA table_info({table_name})')
    cols = cur.fetchall()
    conn.close()
    return '\n'.join([f'{c[1]} ({c[2]})' for c in cols])

print("Tables:", list_tables(DB_PATH))
for t in list_tables(DB_PATH):
    print(f'\nSchema for {t}')
    print(get_schema(DB_PATH, t))


Tables: ['customers', 'order_items', 'orders', 'support_tickets']

Schema for customers
customer_id (INTEGER)
full_name (TEXT)
city (TEXT)
signup_date (TEXT)

Schema for order_items
item_id (INTEGER)
order_id (INTEGER)
product_name (TEXT)
quantity (INTEGER)
unit_price (REAL)

Schema for orders
order_id (INTEGER)
customer_id (INTEGER)
order_date (TEXT)
status (TEXT)
total_amount (REAL)

Schema for support_tickets
ticket_id (INTEGER)
customer_id (INTEGER)
subject (TEXT)
status (TEXT)
created_at (TEXT)


## 6) Build simple SQL tools

These tools are intentionally small and transparent:
- list tables
- show schema
- run a read-only SQL query

This is enough for a guided lab.


In [6]:
from langchain.tools import tool

@tool
def sql_list_tables() -> str:
    """List available tables in the SQLite database."""
    return ", ".join(list_tables(DB_PATH))

@tool
def sql_get_schema(table_name: str) -> str:
    """Return the schema for one table."""
    if table_name not in list_tables(DB_PATH):
        return f"Table not found: {table_name}"
    return get_schema(DB_PATH, table_name)

@tool
def sql_query(query: str) -> str:
    """Run a read-only SQL query and return rows."""
    forbidden = ["insert ", "update ", "delete ", "drop ", "alter ", "create "]
    q_lower = query.lower()
    if any(word in q_lower for word in forbidden):
        return "Only read-only SELECT queries are allowed in this lab."

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    try:
        cur.execute(query)
        rows = cur.fetchall()
        return str(rows)
    except Exception as e:
        return f'SQL error: {e}'
    finally:
        conn.close()

print(sql_list_tables.invoke({}))


C:\Users\MadhiarasanM\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


customers, order_items, orders, support_tickets


## 7) Choose the model backend

This lab supports three common backends:

- Groq
- OpenAI
- Ollama

Use whichever one you have credentials or a local runtime for.


In [7]:
def build_model():
    if MODEL_PROVIDER == 'groq':
        from langchain_groq import ChatGroq
        return ChatGroq(model=MODEL_NAME, temperature=0)

    if MODEL_PROVIDER == 'openai':
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL_NAME, temperature=0)

    if MODEL_PROVIDER == 'ollama':
        from langchain_ollama import ChatOllama
        return ChatOllama(model=MODEL_NAME, temperature=0)

    raise ValueError(f'Unsupported MODEL_PROVIDER={MODEL_PROVIDER}. Use groq, openai, or ollama.')

model = build_model()
print(model)


output_version=None profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x0000023132603A10> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002313280C830> model_name='llama-3.3-70b-versatile' temperature=1e-08 model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None


## 8) Build the SQL agent

LangChain’s SQL-agent guide uses an agent with tools and a system prompt. The prompt tells the agent to inspect tables, inspect schemas, generate a SQLite query, check the query, execute it, and answer from the result. citeturn845823view0


In [8]:
from langchain.agents import create_agent

system_prompt = (
    'You are an assistant designed to interact with a SQLite database.\n\n'
    'Follow these rules:\n'
    '- First list the tables.\n'
    '- Then inspect the schema of only the relevant tables.\n'
    '- Write a syntactically correct SQLite SELECT query.\n'
    '- Keep the query small and focused.\n'
    '- Do not write INSERT, UPDATE, DELETE, DROP, or ALTER statements.\n'
    '- Return the final answer in plain English after using the query result.'
)

tools = [sql_list_tables, sql_get_schema, sql_query]

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
)

print("Agent ready.")


Agent ready.


## 9) Try a few questions

Start with simple questions that map clearly to the mock schema.

Examples:
- How many customers are there?
- Which customers placed completed orders?
- What is the total order amount by city?
- Which support tickets are still open?


In [9]:
questions = [
"How many customers are there?",
"Which customers placed completed orders?",
"What is the total order amount by city?",
]

for q in questions:
    print('\nQUESTION:', q)
    try:
        result = agent.invoke({'messages': [{'role': 'user', 'content': q}]})
        print(result)
    except Exception as e:
        print('Agent error:', e)



QUESTION: How many customers are there?
{'messages': [HumanMessage(content='How many customers are there?', additional_kwargs={}, response_metadata={}, id='5978fb88-93ef-4d05-8416-82aee1c2e4cb'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'r20fz2xr2', 'function': {'arguments': '{}', 'name': 'sql_list_tables'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 407, 'total_tokens': 451, 'completion_time': 0.102259829, 'completion_tokens_details': None, 'prompt_time': 0.034817777, 'prompt_tokens_details': None, 'queue_time': 0.049473008, 'total_time': 0.137077606}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea728-93f7-75f2-9b3e-692c32cfc912-0', tool_calls=[{'name': 'sql_list_tables', 'args': {}, 'id': 'r20fz2xr2', 'type': 'tool_call'}], invalid_tool_calls=[], usage

## 10) A simple fallback path

If you are just learning, you can call the SQL tool directly before using the full agent.

This helps you verify the database behavior first.


In [10]:
print(sql_list_tables.invoke({}))
print(sql_get_schema.invoke({"table_name": "orders"}))
print(sql_query.invoke({"query": "SELECT status, COUNT(*) FROM orders GROUP BY status"}))


customers, order_items, orders, support_tickets
order_id (INTEGER)
customer_id (INTEGER)
order_date (TEXT)
status (TEXT)
total_amount (REAL)
[('completed', 3), ('processing', 1)]


## 11) LangSmith tracing

Enable tracing so you can inspect the agent steps later.

Typical setup:

```env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=text-to-sql-lab
```

The SQL-agent guide recommends using LangSmith to inspect what is happening inside your chain or agent. citeturn845823view0


In [11]:
from langsmith import tracing_context

with tracing_context(enabled=True, project_name=os.getenv('LANGSMITH_PROJECT', 'text-to-sql-lab')):
    try:
        traced_result = agent.invoke({'messages': [{'role': 'user', 'content': 'What is the total order amount by city?'}]})
        print(traced_result)
    except Exception as e:
        print('Tracing example error:', e)


{'messages': [HumanMessage(content='What is the total order amount by city?', additional_kwargs={}, response_metadata={}, id='5203dc07-8485-4038-a5e9-6652acd05fca'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'aadj6jsmq', 'function': {'arguments': '{}', 'name': 'sql_list_tables'}, 'type': 'function'}, {'id': '0zsbcp6wv', 'function': {'arguments': '{"table_name":"orders"}', 'name': 'sql_get_schema'}, 'type': 'function'}, {'id': 'v6c3r8ag3', 'function': {'arguments': '{"table_name":"customers"}', 'name': 'sql_get_schema'}, 'type': 'function'}, {'id': 'r8pj5pmqe', 'function': {'arguments': '{"query":"SELECT c.city, SUM(o.order_amount) FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.city"}', 'name': 'sql_query'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 410, 'total_tokens': 492, 'completion_time': 0.213281063, 'completion_tokens_details': None, 'prompt_time': 0.113877709, 'prompt_token

## 12) Teardown and reset

For a lab, it is useful to wipe the database and re-seed it whenever needed.


In [12]:
def reset_database(db_path: str):
    path = Path(db_path)
    if path.exists():
        path.unlink()
    print('Removed:', path)

# Uncomment to clean up:
reset_database(DB_PATH)


Removed: lab_app.db


## 13) Optional LangSmith custom store note

If you later turn this into a LangGraph or LangSmith deployment, the custom store guide shows how to provide an async context manager that yields a `BaseStore`, and it gives an `AsyncSqliteStore` example. The same guide also notes that SQLite is not recommended for production deployments.


## Key takeaways

- Start with a small local SQLite database.
- Seed a mock dataset that is easy to query.
- Use read-only SQL tools first.
- Ask the model to inspect tables and schemas before generating queries.
- Use LangSmith tracing for visibility.
- Keep the lab simple before moving to deeper agentic workflows.


## References

- SQL agent: https://docs.langchain.com/oss/python/langchain/sql-agent
- LangSmith custom store: https://docs.langchain.com/langsmith/custom-store#define-the-store
